# H1 — WTE excess for AntDir

For each task distribution (TD): collect `(s,a,r,s')`, embed every task, and report
for **both** `(s,a,r)` and `(s,a,s')`:

- **F** — per-task noise floor (mean pairwise distance of a task's embeddings across
  seeds), averaged over the TD's tasks.
- **d(T)** — per-seed mean pairwise distance across the TD's tasks, averaged over seeds.
- **excess = d(T) − F**, for L2 and cosine.

Then the metric-property checks (non-negativity, symmetry, identity, triangle) on all
tasks pooled across TDs. Shared logic lives in `scripts/wte_h1.py`.

Goal-varying (walking direction). Uses hyper `AntDirEnv` (needs mujoco_py). A random ant barely moves, so expect small `(s,a,r)` excess.

In [ ]:
import sys
from pathlib import Path
import numpy as np, pandas as pd

CODE_DIR = Path.cwd().parent if Path.cwd().name == "scripts" else Path.cwd()
HYPER_DIR = CODE_DIR / "hyper" if (CODE_DIR / "hyper").is_dir() else CODE_DIR
for _p in (str(HYPER_DIR), str(CODE_DIR / "scripts")):
    if _p not in sys.path:
        sys.path.insert(0, _p)
import environments  # registers hyper envs
from environments.mujoco.ant_dir import AntDirEnv
import wte_h1 as h1

EP_LEN = 200
N = 5000                                         # tuples / task
SEEDS = list(range(42, 47))                      # 5 data seeds
FEATURE_SETS = ["sar", "sas"]
REWARDS = ["default"]
MAX_SAMPLES = None
DATA = CODE_DIR / "data" / "antdir_h1"

N_TASKS = 8
TASK_DISTS = {
    1: [float(a) for a in np.linspace(np.deg2rad(-15), np.deg2rad(15), N_TASKS)],
    2: [float(a) for a in np.linspace(np.deg2rad(-90), np.deg2rad(90), N_TASKS)],
    3: [float(a) for a in np.linspace(-np.pi, np.pi, N_TASKS, endpoint=False)],
}
ALL_TASKS = [t for ts in TASK_DISTS.values() for t in ts]

def _reset(env, seed=None):
    """hyper AntDir is old-gym style (obs) or (obs, info)."""
    out = env.reset() if seed is None else env.reset(seed=seed)
    return out[0] if isinstance(out, tuple) else out

def _step(env, a):
    """Returns (nobs, reward, done). Handles 4- or 5-tuple step."""
    out = env.step(a)
    if len(out) == 5:
        nobs, r, term, trunc, _ = out
        return nobs, r, bool(term or trunc)
    nobs, r, done, _ = out
    return nobs, r, bool(done)

_probe = AntDirEnv(max_episode_steps=EP_LEN)
_o = _reset(_probe)
SPEC = dict(obs_dim=int(np.ravel(_o).shape[0]),
            a_cols=int(_probe.action_space.shape[0]),
            discrete=False, n_actions=None)

def _dir(reward, seed):
    return DATA / f"seed{seed}"

def collect(reward, seed):
    out = _dir(reward, seed); out.mkdir(parents=True, exist_ok=True)
    rng = np.random.default_rng(seed)
    env = AntDirEnv(max_episode_steps=EP_LEN)
    for ang in ALL_TASKS:
        f = out / f"dir_{ang:.6f}.npy"
        if f.exists():
            continue
        env.set_task(ang)
        obs = _reset(env)
        rows = []
        while len(rows) < N:
            a = rng.uniform(env.action_space.low, env.action_space.high)
            nobs, r, done = _step(env, a)
            rows.append(np.concatenate([np.ravel(obs), np.ravel(a), [r], np.ravel(nobs)]))
            obs = nobs
            if done:
                env.set_task(ang); obs = _reset(env)
        np.save(f, np.asarray(rows[:N], dtype=np.float64))

def load_fn_for(reward):
    def load(task, seed):
        return np.load(_dir(reward, seed) / f"dir_{task:.6f}.npy")
    return load

In [ ]:
# Collect (s,a,r,s') for every task over all seeds. Safe to re-run: skips files.
for _reward in REWARDS:
    for _seed in SEEDS:
        collect(_reward, _seed)
print("collection done:", {r: len(SEEDS) for r in REWARDS}, "seeds x",
      sum(len(v) for v in TASK_DISTS.values()), "tasks")

In [ ]:
# Embed, then per TD: F (mean per-task floor), d(T) (mean pairwise over seeds),
# and excess = d(T) - F, for both L2 and cosine, for (s,a,r) and (s,a,s').
EMB, _tables = {}, []
for _reward in REWARDS:
    _load = load_fn_for(_reward)
    _mu, _sd = h1.pooled_reward_stats(_load, ALL_TASKS, SEEDS,
                                      SPEC["obs_dim"], SPEC["a_cols"])
    for _fs in FEATURE_SETS:
        _emb = h1.compute_embeddings(
            TASK_DISTS, SEEDS, _load, SPEC["obs_dim"], SPEC["a_cols"], _fs, _mu, _sd,
            discrete=SPEC["discrete"], n_actions=SPEC["n_actions"],
            max_samples=MAX_SAMPLES)
        EMB[(_reward, _fs)] = _emb
        _t = h1.summarize(TASK_DISTS, SEEDS, _emb)
        _t.insert(0, "features", _fs)
        _t.insert(0, "reward", _reward)
        _tables.append(_t)
results = pd.concat(_tables, ignore_index=True)
results.round(4)

In [ ]:
# Metric-property checks on all tasks pooled across TDs (pairwise matrix averaged
# over seeds; identity threshold delta = mean same-task-across-seeds distance).
_cls = [h1.classify(TASK_DISTS, SEEDS, _emb).assign(reward=_r, features=_f)
        for (_r, _f), _emb in EMB.items()]
classification = pd.concat(_cls, ignore_index=True)[[
    "reward", "features", "metric", "delta", "nonneg", "symmetric",
    "identity_ok", "n_collapsed", "n_pairs", "triangle_ok", "triangle_bad", "class"]]
classification.round(4)